# Homework 05 — Data Storage

**Anna Cui** · FRE 5040

Adapted from `stage05_data-storage_homework-starter.ipynb`. A reproducible
storage layer: env-driven paths, CSV and Parquet round trips, reload
validation, and IO utilities that route by file suffix.

> **Reproducibility = (organized folders) x (env-driven paths) x (explicit
> save/load) x (validation).** Multiplication, not addition.

In [1]:
from pathlib import Path
ROOT = Path.cwd()
for rel in ['.env', '.env.example']:
    print(f"  [{'OK ' if (ROOT / rel).exists() else 'MISS'}]  {rel}")
print('\nLooking in:', ROOT)

  [OK ]  .env
  [OK ]  .env.example

Looking in: /Users/annacui/NYU/Bootcamp/Bootcamp 4/bootcamp_Anna_Cui/homework/homework05


In [2]:
import os, pathlib, datetime as dt, json, typing as t

import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
RAW = pathlib.Path(os.getenv('DATA_DIR_RAW', 'data/raw'))
PROC = pathlib.Path(os.getenv('DATA_DIR_PROCESSED', 'data/processed'))
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
print('RAW  ->', RAW.resolve())
print('PROC ->', PROC.resolve())

RAW  -> /Users/annacui/NYU/Bootcamp/Bootcamp 4/bootcamp_Anna_Cui/homework/homework05/data/raw
PROC -> /Users/annacui/NYU/Bootcamp/Bootcamp 4/bootcamp_Anna_Cui/homework/homework05/data/processed


## 1. Sample data — and the starter's missing seed

The starter builds `150 + np.random.randn(20).cumsum()` with **no seed**, so
every run produces different prices under a different timestamped filename.
Two runs, two datasets, no way to tell which file holds which. The lecture
seeds deliberately for exactly this reason.

Seeding is the difference between a file you can regenerate and one you can
only preserve.

In [3]:
np.random.seed(5)          # the starter omits this; without it nothing reproduces

dates = pd.date_range('2024-01-01', periods=20, freq='D')
df = pd.DataFrame({
    'date': dates,
    'ticker': ['VTI'] * 20,
    'price': 150 + np.random.randn(20).cumsum(),
})
print(df.dtypes.to_dict())
df.head()

{'date': dtype('<M8[ns]'), 'ticker': dtype('O'), 'price': dtype('float64')}


,date,ticker,price
0,2024-01-01,VTI,150.441227
1,2024-01-02,VTI,150.110357
2,2024-01-03,VTI,152.541129
3,2024-01-04,VTI,152.289036
4,2024-01-05,VTI,152.398646


## 2. Save in two formats

CSV to `data/raw/`, Parquet to `data/processed/`. The Parquet write is wrapped
because it depends on an engine that may not be installed — the one place here
where `try`/`except` is load-bearing rather than decorative.

In [4]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M')

stamp = ts()                       # one stamp for both files, so they pair up
csv_path = RAW / f"sample_{stamp}.csv"
df.to_csv(csv_path, index=False)
print('Saved CSV     ->', csv_path)

pq_path = PROC / f"sample_{stamp}.parquet"
try:
    df.to_parquet(pq_path)
    print('Saved Parquet ->', pq_path)
except Exception as exc:
    print('Parquet engine not available - install pyarrow or fastparquet.')
    print('Error:', exc)
    pq_path = None

Saved CSV     -> data/raw/sample_20260828-1643.csv
Saved Parquet -> data/processed/sample_20260828-1643.parquet


## 3. Reload and validate

**The question is never "did a file get written?" — it is "is the data still
there when I read it back?"** Both files exist. Only one preserved the dtypes.

In [5]:
def validate_loaded(original, reloaded, cols=('date', 'ticker', 'price')):
    checks = {
        'shape_equal': original.shape == reloaded.shape,
        'cols_present': all(col in reloaded.columns for col in cols),
    }
    if 'price' in reloaded.columns:
        checks['price_is_numeric'] = pd.api.types.is_numeric_dtype(reloaded['price'])
    if 'date' in reloaded.columns:
        checks['date_is_datetime'] = pd.api.types.is_datetime64_any_dtype(reloaded['date'])
    checks['values_match'] = bool(np.allclose(original['price'], reloaded['price']))
    return checks

# Deliberately WITHOUT parse_dates, to show what CSV actually returns.
naive_csv = pd.read_csv(csv_path)
print('CSV, no parse_dates :', validate_loaded(df, naive_csv))
print('   date came back as:', naive_csv['date'].dtype)

CSV, no parse_dates : {'shape_equal': True, 'cols_present': True, 'price_is_numeric': True, 'date_is_datetime': False, 'values_match': True}
   date came back as: object


In [6]:
df_csv = pd.read_csv(csv_path, parse_dates=['date'])
print('CSV, parse_dates    :', validate_loaded(df, df_csv))

if pq_path:
    df_pq = pd.read_parquet(pq_path)
    print('Parquet             :', validate_loaded(df, df_pq))
    print('   date came back as:', df_pq['date'].dtype, '- no instruction needed')

CSV, parse_dates    : {'shape_equal': True, 'cols_present': True, 'price_is_numeric': True, 'date_is_datetime': True, 'values_match': True}
Parquet             : {'shape_equal': True, 'cols_present': True, 'price_is_numeric': True, 'date_is_datetime': True, 'values_match': True}
   date came back as: datetime64[ns] - no instruction needed


**That contrast is the stage.** CSV has no type system: `date` returns as
`object` unless the reader is told otherwise, and a reader who forgets carries
text into a time-series model. Parquet stores the schema with the data, so the
dtype survives without anyone remembering anything.

## 4. IO utilities — the starter's, improved

**`read_df` read the file twice.** The starter called `pd.read_csv(p, nrows=0)`
to inspect the header, then read the whole file again. Now the header is read
once and reused.

**`read_df` only parsed a column literally named `date`.** A `trade_date`
column silently returned as text — the "silent type drift" the reading warns
about. Now any column whose name ends in `date` is parsed.

**`write_df` refuses to write an empty frame.** Saving zero rows succeeds
silently and produces a file that looks fine until something downstream
returns nothing.

In [7]:
def detect_format(path: t.Union[str, pathlib.Path]) -> str:
    s = str(path).lower()
    if s.endswith('.csv'):
        return 'csv'
    if s.endswith(('.parquet', '.pq', '.parq')):
        return 'parquet'
    raise ValueError(f'Unsupported format: {path}')


def write_df(df: pd.DataFrame, path: t.Union[str, pathlib.Path]) -> pathlib.Path:
    if df.empty:
        raise ValueError('refusing to write an empty DataFrame')
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    if detect_format(p) == 'csv':
        df.to_csv(p, index=False)
    else:
        try:
            df.to_parquet(p)
        except ImportError as exc:
            raise RuntimeError('Parquet engine not available. '
                               'Install pyarrow or fastparquet.') from exc
    return p


def read_df(path: t.Union[str, pathlib.Path]) -> pd.DataFrame:
    p = pathlib.Path(path)
    if detect_format(p) == 'csv':
        header = pd.read_csv(p, nrows=0).columns          # read the header ONCE
        date_cols = [col for col in header if str(col).lower().endswith('date')]
        return pd.read_csv(p, parse_dates=date_cols or None)
    try:
        return pd.read_parquet(p)
    except ImportError as exc:
        raise RuntimeError('Parquet engine not available. '
                           'Install pyarrow or fastparquet.') from exc

In [8]:
demo = df.rename(columns={'date': 'trade_date'})
p_csv = write_df(demo, RAW / f"util_{ts()}.csv")
p_pq = write_df(demo, PROC / f"util_{ts()}.parquet")

back_csv, back_pq = read_df(p_csv), read_df(p_pq)
print('csv  trade_date ->', back_csv['trade_date'].dtype, '(starter left this as text)')
print('pq   trade_date ->', back_pq['trade_date'].dtype)

csv  trade_date -> datetime64[ns] (starter left this as text)
pq   trade_date -> datetime64[ns]


In [9]:
for call, label in [
    (lambda: write_df(pd.DataFrame(), RAW / 'empty.csv'), 'write_df on an empty frame'),
    (lambda: detect_format('notes.txt'),                  'detect_format on .txt'),
]:
    try:
        call()
    except ValueError as exc:
        print(f"{label:<32} -> ValueError: {exc}")

write_df on an empty frame       -> ValueError: refusing to write an empty DataFrame
detect_format on .txt            -> ValueError: Unsupported format: notes.txt


## 5. What a round trip actually costs — nested data

A record whose `notes` field is a dict containing a list. Both saves succeed.
**Neither raises. Two success messages, and one file has lost the data.**

In [10]:
unstructured = [
    {'date': '2024-01-01', 'ticker': 'VTI', 'price': 150.0,
     'notes': {'summary': 'Stable', 'details': ['No major events', 'Market steady']}},
    {'date': '2024-01-02', 'ticker': 'VTI', 'price': 151.2,
     'notes': {'summary': 'Slight increase', 'details': ['Broad rally']}},
]

stamp2 = ts()
with open(RAW / f"unstructured_{stamp2}.json", 'w') as fh:
    json.dump(unstructured, fh, indent=2)

nested = pd.DataFrame(unstructured)
csv_file = RAW / f"nested_{stamp2}.csv"
nested.to_csv(csv_file, index=False)

pq_file = PROC / f"nested_{stamp2}.parquet"
try:
    nested.to_parquet(pq_file)
    pq_ok = True
except Exception as exc:
    print('Parquet write of nested data failed:', exc)
    pq_ok = False

print('both saves reported success:', csv_file.exists(), pq_ok)

both saves reported success: True True


In [11]:
back_csv_nested = pd.read_csv(csv_file)
print('original      notes[0] is a', type(unstructured[0]['notes']).__name__)
print('from CSV      notes[0] is a', type(back_csv_nested['notes'][0]).__name__)

try:
    print('CSV     - ask it for the summary:', back_csv_nested['notes'][0]['summary'])
except TypeError as exc:
    print('CSV     - ask it for the summary: TypeError:', exc)
    print('          what came back was text:', repr(back_csv_nested['notes'][0])[:60])

if pq_ok:
    back_pq_nested = pd.read_parquet(pq_file)
    print('from Parquet  notes[0] is a', type(back_pq_nested['notes'][0]).__name__)
    print('Parquet - ask it for the summary:', back_pq_nested['notes'][0]['summary'])
    was = unstructured[0]['notes']['details']
    now = back_pq_nested['notes'][0]['details']
    print('details went in a', type(was).__name__, 'and came back a', type(now).__name__,
          '- same values:', list(now) == list(was))

original      notes[0] is a dict
from CSV      notes[0] is a str
CSV     - ask it for the summary: TypeError: string indices must be integers
          what came back was text: "{'summary': 'Stable', 'details': ['No major events', 'Marke
from Parquet  notes[0] is a dict
Parquet - ask it for the summary: Stable
details went in a list and came back a ndarray - same values: True


**Conclusions.** CSV is a flat text format: it flattened the dictionary and
told us nothing. Parquet kept the structure and the values, **but not the exact
Python types** — the inner list came back as a numpy array, close enough to
work with and not close enough for a naive `==`.

**Verify a round trip, never assume one.** The raw JSON stays in `data/raw/` as
the source of truth: you can always re-derive a table, but you cannot re-derive
structure you already threw away.

## Data Storage

*(Mirrored into `README.md`.)*

| Folder | Holds | Rule |
|---|---|---|
| `data/raw/` | Inputs exactly as acquired, plus the source JSON | **Immutable.** Never edited by hand; a new version gets a new filename |
| `data/processed/` | Derived tables written by code | Deletable and re-creatable by re-running this notebook |

**Formats, and why.** CSV in `raw/` — human-readable, diffable in git,
universally accepted; costs no schema, no types, no nesting. Parquet in
`processed/` — columnar, compressed, preserves dtypes, fast reads; costs a
binary format and an engine dependency. Rule of thumb: CSV for small exchange
files, Parquet for analysis-ready tables.

**How the code reads and writes.** Paths come from `.env` via `os.getenv`,
never hardcoded: `DATA_DIR_RAW=data/raw` and
`DATA_DIR_PROCESSED=data/processed`. `write_df` and `read_df` route on the file
suffix, create missing parent directories, refuse to write an empty frame,
raise a clear message when the Parquet engine is absent, and parse any column
whose name ends in `date`.

**Validation on reload:** shape equality, required columns present, `price`
numeric, `date` datetime, values numerically equal to the original.

In [12]:
for folder in (RAW, PROC):
    print(f"\n{folder}:")
    for path in sorted(folder.iterdir()):
        if path.name != '.gitkeep':
            print(f"  {path.name:<40} {path.stat().st_size:>9,} bytes")


data/raw:
  nested_20260828-1643.csv                       200 bytes
  sample_20260828-1643.csv                       694 bytes
  unstructured_20260828-1643.json                386 bytes
  util_20260828-1643.csv                         700 bytes

data/processed:
  nested_20260828-1643.parquet                 3,701 bytes
  sample_20260828-1643.parquet                 3,016 bytes
  util_20260828-1643.parquet                   3,064 bytes
